In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu transformers sentence-transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 146.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
import os
import math
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import PromptTemplate
from transformers import GenerationConfig



In [5]:
import os
from google.colab import userdata


hf_login = userdata.get('hf_login')

os.environ["MY_API_KEY"] = hf_login

from huggingface_hub import login


login(hf_login)

In [6]:
VECTOR_DB_FOLDER = "/content/drive/MyDrive/Graduation Project/Vector DataBase"
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # Best for Colab Free
# MODEL_NAME = "inceptionai/jais-adapted-7b-chat"  # Best Arabic model

MODEL_NAME = "QCRI/Fanar-1-9B-Instruct"


In [7]:
# Load Embeddings
print("Loading embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="medmediani/Arabic-KW-Mdel")


Loading embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Load all Vector DBs with metadata indexing
print("Loading vector databases...")
all_dbs = []
db_metadata_index = {}  # Fast lookup: {law_type: [db_indices]}

folders = ["مدني", "قانون المرافعات"]
for folder in folders:
    folder_path = os.path.join(VECTOR_DB_FOLDER, folder)
    if not os.path.exists(folder_path):
        print(f"Warning: Folder not found: {folder_path}")
        continue
    for subfolder in os.listdir(folder_path):
        db_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(db_path) and 'index.faiss' in os.listdir(db_path):
            db = FAISS.load_local(db_path, embeddings, allow_dangerous_deserialization=True)
            db_idx = len(all_dbs)
            all_dbs.append((db, subfolder, folder))  # Added folder (law type)

            # Index by law type for fast filtering
            if folder not in db_metadata_index:
                db_metadata_index[folder] = []
            db_metadata_index[folder].append(db_idx)

print(f"Loaded {len(all_dbs)} databases from {len(db_metadata_index)} law types.")
print(f"Law types: {list(db_metadata_index.keys())}")


Loading vector databases...
Loaded 99 databases from 2 law types.
Law types: ['مدني', 'قانون المرافعات']


In [9]:
# Load Qwen2.5 LLM and Tokenizer
print(f"Loading {MODEL_NAME} model... (this may take a while)")


model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        # torch_dtype="auto",
        torch_dtype= torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", trust_remote_code=True)

print("Model loaded successfully!")

Loading QCRI/Fanar-1-9B-Instruct model... (this may take a while)


config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/18.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

Model loaded successfully!


In [10]:
# Prompt Template for RAG + LLM
prompt_template = """أنت مساعد قانوني متخصص في القانون المصري.
مهمتك تقديم إجابة قانونية دقيقة، منضبطة، وقابلة للاعتماد الأكاديمي.

 قواعد إلزامية:
1. استخدم فقط المعلومات الواردة صراحة في السياق القانوني المقدم.
2. لا تضف أي قاعدة أو رأي أو مثال غير مستفاد من السياق.
3. إذا لم يتضمن السياق نصًا صريحًا للإجابة:
   - صرّح بذلك بوضوح.
   - ويمكنك فقط تقديم استنتاج قانوني مشروط بصيغة: "يمكن استنتاج ذلك في ضوء القواعد العامة إذا كان..."
4. يُمنع منعًا باتًا:
   - اختلاق مواد قانونية أو أحكام محكمة النقض.
   - إدخال تشريعات غير مصرية.
   - استخدام لغة غير العربية.

 منهج الإجابة:
- ابدأ بتعريف قانوني دقيق (إن وجد في السياق).
- ثم بيّن الشروط أو الضوابط في نقاط واضحة.
- ثم الحالات أو الاستثناءات (إن وردت).
- اختم ببيان نوع القانون (مدني / مرافعات / تجاري…).

 السياق القانوني:
{context}

 السؤال:
{question}

 الإجابة:
"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])


In [11]:
# VRAM check
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")


GPU: Tesla T4
Total VRAM: 14.74 GB
Allocated: 12.81 GB


In [12]:
def get_candidate_dbs(all_dbs, law_type=None):
    """
    Select only relevant vector databases based on law type.
    """
    indices = []

    for idx, (_, _, db_law_type) in enumerate(all_dbs):
        if law_type and db_law_type != law_type:
            continue
        indices.append(idx)

    return indices

In [13]:
def search_with_metadata_filter(
    query,
    all_dbs,
    law_type=None,
    base_top_k=6,
    similarity_threshold=0.15
):
    """
    Optimized semantic search using:
    - Metadata pre-filtering
    - FAISS similarity search
    - Lightweight metadata re-ranking
    """

    results = []

    # 🔹 Step 1: Pre-filter DBs
    db_indices = get_candidate_dbs(all_dbs, law_type)

    if not db_indices:
        return []

    # 🔹 Step 2: Dynamic top_k
    top_k = 3 if law_type else base_top_k

    # 🔹 Step 3: FAISS Search
    for idx in db_indices:
        db, db_name, db_law_type = all_dbs[idx]

        docs_with_scores = db.similarity_search_with_score(query, k=top_k)

        for doc, distance in docs_with_scores:
            # FAISS uses L2 distance → convert to similarity
            similarity = 1 / (1 + distance)

            # if similarity < similarity_threshold:
            #     continue

            results.append((doc, db_name, db_law_type, similarity))

    if not results:
        return []

    # --------------------------------------------------
    # 4) Metadata-aware Re-ranking (Cheap & Effective)
    # --------------------------------------------------
    reranked = []

    for doc, db_name, law_type, sim in results:
        boost = 0.0

        topic = doc.metadata.get("topic", "")
        book = doc.metadata.get("book_title", "")

        if topic and topic in query:
            boost += 0.10

        if book and book in query:
            boost += 0.10

        reranked.append((doc, db_name, law_type, sim + boost))

    reranked.sort(key=lambda x: x[3], reverse=True)

    return reranked[:top_k]


In [14]:
# Function to Generate Response with Qwen2.5
def generate_response(query, context):
    """
    Generate a response using the Qwen2.5 model.

    Args:
        query: User's question
        context: Retrieved context from vector database

    Returns:
        Generated response string
    """
    messages = [
        {"role": "system", "content": "أنت مساعد قانوني مفيد للقانون المصري."},
        {"role": "user", "content": PROMPT.format(context=context, question=query)}
    ]


    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # ضبط التوليد ليكون دقيقاً جداً (Low Temperature)
    gen_config = GenerationConfig(
        max_new_tokens=1024,
        temperature=0.1,    # حرارة منخفضة جداً لتقليل الإبداع الوهمي
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1
    )

    generated_ids = model.generate(**model_inputs, generation_config=gen_config)

    # استخراج الإجابة فقط
    response_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    return tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

In [15]:
def build_context(results, max_docs=3, max_total_chars=4000):
    context_blocks = []
    current_chars = 0

    for doc, _, _, _ in results[:max_docs]:
        header = f"""
المصدر:
- الكتاب: {doc.metadata.get('book_title', 'غير متوفر')}
- المؤلف: {doc.metadata.get('author', 'غير متوفر')}
- المجال: {doc.metadata.get('law_area', 'غير محدد')}
"""
        block = header + "\n" + doc.page_content
        if current_chars + len(block) > max_total_chars:
            break
        context_blocks.append(block)
        current_chars += len(block)

    return "\n\n---\n\n".join(context_blocks) if context_blocks else "لا توجد سياقات كافية."

In [16]:
def prototype_chat():
    """
    Interactive chat loop for the legal assistant with metadata filtering.
    """
    print("\n" + "="*50)
    print("مرحباً بك في المساعد القانوني!")
    print("Type 'exit' or 'خروج' to quit.")
    print("="*50 + "\n")

    # Show available law types
    print("أنواع القوانين المتاحة ")
    for i, law_type in enumerate(db_metadata_index.keys(), 1):
        print(f"{i}. {law_type}")
    print()

    while True:
        query = input("\nسؤالك: ").strip()

        if query.lower() in ['exit', 'خروج', 'quit']:
            print("شكراً لاستخدامك المساعد القانوني. وداعاً!")
            break

        if not query:
            print("من فضلك أدخل سؤالاً .")
            continue

        law_type = None
        # Search with metadata filtering
        print("\nجاري البحث... / Searching...")
        results = search_with_metadata_filter(query, all_dbs, law_type=law_type)

        if not results:
            print("لم يتم العثور على نتائج ذات صلة.")
            continue


        # Prepare context from retrieved documents
        context = build_context(results)


        # Generate response
        print("جاري إنشاء الإجابة... ")
        response = generate_response(query, context)

        # Display response
        print("\n" + "="*50)
        print("الإجابة / Response:")
        print("-"*50)
        print(response)
        print("="*50)

        # Display sources (only law type + clean file name)
        print("\nالمصادر / Sources:")
        for i, (doc, _, db_law_type, _) in enumerate(results):
            print(
                f"{i+1}. [{db_law_type}] "
                f"{doc.metadata.get('book_title', 'غير متوفر')} – "
                f"{doc.metadata.get('author', 'غير متوفر')}"
            )


In [18]:
if __name__ == "__main__":


    # Start the chat interface
    start_chat = input("\nهل تريد بدء المحادثة؟ (y/n): ").strip().lower()
    if start_chat in ['y', 'yes', 'نعم', '']:
        prototype_chat()
    else:
        print("وداعاً! / Goodbye!")


هل تريد بدء المحادثة؟ (y/n): y

مرحباً بك في المساعد القانوني!
Type 'exit' or 'خروج' to quit.

أنواع القوانين المتاحة 
1. مدني
2. قانون المرافعات


سؤالك: ما هى شروط اتحاد الملاك

جاري البحث... / Searching...
جاري إنشاء الإجابة... 

الإجابة / Response:
--------------------------------------------------
وفقا للمعلومات المقدمة من المصدر المذكور ("الموسوعة النموذجية في الملكية العقارية في ضوء الفقه وقضاء النقض" للسيد عبد الوهاب عرفة)، فإن شروط اتحاد الملاك تتضمن ما يلي:

1. يجب أن يكون اتحاد الملاك جمعية من جميع ملاك الطبقات والشقق في البناء الواحد.
2. يكتسب اتحاد الملاك شخصية معنوية بموجب القانون.
3. يتم تأسيس اتحاد الملاك عبر إجراءات محددة، بما في ذلك:
    * كتابة عقد تأسيس بين الأعضاء المؤسسين.
    * تحديد مقر الاتحاد ومجلس إدارته.
    * وضع نظام داخلي للاتحاد.
    * الحصول على موافقة الجهات المعنية مثل حي المنطقة.
4. يجب فتح حساب بنكي باسم الاتحاد لتسهيل التعاملات المالية.
5. يجب حفظ وثائق وأوراق الاتحاد بشكل منظم داخل ملف خاص.

ومع ذلك، ينبغي ملاحظة أنه لا يوجد ذكر صريح لشروط محددة 

✅ السؤال 1 (مدني – مصادر الالتزام)

❓ السؤال:
ما هي مصادر الالتزام في القانون المدني المصري؟

✅ الإجابة المتوقعة:
مصادر الالتزام في القانون المدني المصري هي:

العقد

الإرادة المنفردة في الحالات التي يقررها القانون

العمل غير المشروع

الإثراء بلا سبب

القانون

وقد نظم المشرع هذه المصادر في التقنين المدني المصري، وشرحها الفقه مثل السنهوري ونبيل إبراهيم سعد باعتبارها الأساس القانوني لقيام الالتزام.

📚 مصادر متوقعة من DB:

النظرية العامة للالتزام – مصادر الالتزام

التقنين المدني المصري

كتاب مصادر الالتزام المدني

✅ السؤال 2 (مدني – آثار الالتزام)

❓ السؤال:
ما المقصود بتنفيذ الالتزام عينًا؟ ومتى يتحول إلى تعويض؟

✅ الإجابة المتوقعة:
تنفيذ الالتزام عينًا هو قيام المدين بتنفيذ ما التزم به تنفيذًا مطابقًا لما ورد في العقد أو القانون.
ويتحول التنفيذ إلى تعويض إذا:

أصبح التنفيذ العيني مستحيلًا

أو كان مرهقًا للمدين

أو تأخر المدين في التنفيذ دون مبرر

وفي هذه الحالة يُلزم المدين بتعويض الدائن عن الضرر وفقًا لأحكام المسئولية العقدية.

📚 مصادر متوقعة:

نظرية الالتزام – آثار الالتزام

أحكام الالتزام

التقنين المدني المصري

✅ السؤال 3 (مدني – العقود)

❓ السؤال:
ما الفرق بين العقد الرضائي والعقد الشكلي في القانون المدني؟

✅ الإجابة المتوقعة:
العقد الرضائي ينعقد بمجرد توافق الإيجاب والقبول دون اشتراط شكل خاص، مثل عقد البيع بوجه عام.
أما العقد الشكلي فلا ينعقد إلا باستيفاء شكل معين يحدده القانون، كالتسجيل أو الكتابة الرسمية، مثل بيع العقارات في بعض الحالات.

والهدف من الشكلية هو حماية المتعاقدين وتحقيق الاستقرار القانوني.

📚 مصادر متوقعة:

العقود التي تقع على الملكية

عقد البيع

المطول في القانون المدني

✅ السؤال 4 (مرافعات – الدفوع)

❓ السؤال:
ما الفرق بين الدفع الشكلي والدفع الموضوعي في قانون المرافعات؟

✅ الإجابة المتوقعة:
الدفع الشكلي يوجه إلى إجراءات الخصومة، ويهدف إلى عدم قبول الدعوى أو بطلانها دون التعرض للحق الموضوعي، مثل الدفع بعدم الاختصاص.

أما الدفع الموضوعي فيتعلق بأصل الحق المدعى به، ويهدف إلى إنكاره أو نفيه، مثل الدفع بانقضاء الالتزام أو عدم وجوده.

📚 مصادر متوقعة:

الدفوع المدنية

موسوعة الدفوع المدنية

شرح قانون المرافعات

✅ السؤال 5 (مرافعات – الطعن بالنقض)

❓ السؤال:
ما هي أسباب الطعن بالنقض في المواد المدنية؟

✅ الإجابة المتوقعة:
يُقبل الطعن بالنقض في الأحكام المدنية إذا شابه:

مخالفة القانون أو الخطأ في تطبيقه

بطلان في الحكم أو في الإجراءات

القصور في التسبيب

التناقض في الأسباب

ويعد الطعن بالنقض طريقًا استثنائيًا يهدف إلى مراقبة صحة تطبيق القانون وليس إعادة نظر الموضوع.

📚 مصادر متوقعة:

النقض المدني

المرافعات المدنية والتجارية

بحوث وتعليقات على الأحكام

🔍 كيف تستخدم الأسئلة دي عمليًا؟
أثناء الاختبار:

اسأل نفس السؤال بصياغات مختلفة

جرّب:

بدون تحديد نوع القانون

مع اختيار (مدني / مرافعات)

راقب:

هل الإجابة دقيقة؟

هل ذكر نوع القانون؟

هل استخدم سياق حقيقي من الكتب؟